# ChatBot EcoMarket — v1.1.2
**Migración: Excel → PostgreSQL**

| Campo | Valor |
|---|---|
| Versión | 1.1.2 |
| Base anterior | ChatBot_1.0.5.ipynb (Excel) |
| Motor de datos | PostgreSQL 18 / pgAdmin 4 |
| Arquitectura | `RetailService` como única capa de acceso a SQL |

### Changelog vs 1.0.5
- **Fuente de datos**: Excel → PostgreSQL (`ChatBot_Ecomarket`)
- **Logging**: CSV local → tabla `logs_chatbot` en PostgreSQL
- **Acceso a datos**: funciones sueltas → clase `RetailService`
- **Credenciales**: variables de entorno / archivo `.env`
- **Validación de esquema**: consulta automática a `information_schema`
- **Queries**: 100 % parametrizadas (sin riesgo de SQL injection)
- **NLP**: mismo modelo `mDeBERTa-v3` zero-shot, sin cambios

### Secciones
1. Configuración  
2. Conexión PostgreSQL  
3. RetailService  
4. Utilidades NLP  
5. Reglas de intención  
6. Generación de respuestas  
7. Logging del chatbot  
8. Tests conversacionales  
9. Chat interactivo


## 1. Configuración

In [1]:
!pip install psycopg2-binary python-dotenv transformers torch

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
# ============================================================
# 1. CONFIGURACION GENERAL
# ============================================================
# Dependencias: pip install psycopg2-binary python-dotenv transformers torch
# Credenciales: crea un archivo .env en la misma carpeta que este notebook:
#
#   DB_HOST=localhost
#   DB_PORT=5432
#   DB_NAME=ChatBot_Ecomarket
#   DB_USER=postgres
#   DB_PASSWORD=tu_password
#
# NUNCA subas el .env al repositorio. Añádelo a .gitignore.
# ============================================================

import os
import re
from pathlib import Path
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional, List, Dict, Any

# python-dotenv carga .env si existe; si no, usa las variables de entorno del sistema
try:
    from dotenv import load_dotenv
    env_path = None
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base / ".env", base / "Bot" / ".env", base / "ChatBot_EcoMarket" / "Bot" / ".env"):
            if candidate.exists():
                env_path = candidate
                break
        if env_path:
            break

    if env_path:
        load_dotenv(env_path, override=True)
        print(f"✓ .env cargado desde {env_path}")
    else:
        print("⚠ No se encontró .env. Usando variables de entorno del sistema.")
except ImportError:
    print("⚠ python-dotenv no instalado. Usando variables de entorno del sistema.")

# ── Conexión PostgreSQL ──────────────────────────────────────────────────────
DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = int(os.getenv("DB_PORT", "5432"))
DB_NAME     = os.getenv("DB_NAME",     "ChatBot_Ecomarket")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if not DB_PASSWORD.strip():
    print("⚠ DB_PASSWORD está vacío. Revisa/crea el archivo .env junto al notebook.")

# ── NLP ─────────────────────────────────────────────────────────────────────
MODEL_NAME           = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
MODEL_LOCAL_ONLY     = False
CONFIDENCE_THRESHOLD = 0.55
INTENCION_SOPORTE    = "Hablar con soporte humano"

ETIQUETAS_NEGOCIO = [
    "Estado del pedido",
    "Incidencia con pedido incompleto o no recibido",
    "Devoluciones y cambios",
    "Producto dañado caducado o en mal estado",
    "Disponibilidad de productos",
    "Promociones y cupones",
    "Métodos de pago y compra",
    INTENCION_SOPORTE,
]

# ── Negocio ──────────────────────────────────────────────────────────────────
VERSION               = "1.1.2"
CANTIDAD_POR_DEFECTO  = 1
COMANDOS_SALIDA       = {"salir", "escape", "exit", "quit", "cerrar"}

POLITICAS = {
    "devoluciones": (
        "Las devoluciones pueden solicitarse hasta 30 días después de la compra, "
        "presentando ticket o número de pedido. En productos frescos, perecederos "
        "o de higiene pueden aplicar restricciones."
    ),
    "promociones": (
        "Las promociones y cupones dependen de sus condiciones. Algunas ofertas "
        "pueden ser exclusivas de tienda física o del e-commerce."
    ),
    "pagos": (
        "Aceptamos tarjeta de crédito, tarjeta de débito, PayPal y los métodos "
        "disponibles durante el checkout online."
    ),
}

# ── Tablas mínimas requeridas ─────────────────────────────────────────────────
TABLAS_REQUERIDAS = [
    "categorias", "productos", "inventario",
    "clientes", "pedidos", "detalle_pedidos",
    "movimientos_inventario", "logs_chatbot",
]

print(f"ChatBot EcoMarket v{VERSION} — configuración cargada")
print(f"DB target: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


✓ .env cargado desde c:\data_sciences\GIT\ChatBot_EcoMarket\ChatBot_EcoMarket\Bot\.env
ChatBot EcoMarket v1.1.2 — configuración cargada
DB target: postgres@localhost:5432/ChatBot_Ecomarket


## 2. Conexión PostgreSQL

In [3]:
# ============================================================
# 2. CONEXION POSTGRESQL
# ============================================================

import psycopg2
from psycopg2.extras import RealDictCursor

def get_connection():
    """Devuelve una conexión nueva a PostgreSQL. Lanza excepción si falla."""
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        connect_timeout=10,
    )

def probar_conexion():
    """Verifica que la conexión a la base de datos es posible."""
    try:
        with get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT version();")
                version = cur.fetchone()[0]
        print(f"✓ Conexión OK → {version[:60]}")
        return True
    except psycopg2.OperationalError as exc:
        print(f"✗ Error de conexión: {exc}")
        print("  Revisa DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD en tu .env")
        return False

probar_conexion()


✓ Conexión OK → PostgreSQL 18.0 on x86_64-windows, compiled by msvc-19.44.35


True

## 3. RetailService
Capa única de acceso a PostgreSQL. El chatbot nunca escribe SQL directamente.

In [4]:
# ============================================================
# 3. RETAIL SERVICE
# Toda consulta o escritura a PostgreSQL pasa por esta clase.
# El notebook solo llama a métodos de RetailService.
# ============================================================

class RetailService:
    """
    Capa de acceso a datos para EcoMarket sobre PostgreSQL.
    Usa consultas parametrizadas en todos los métodos para evitar SQL injection.
    """

    # ── Cache en memoria (se recarga con refresh_cache) ──────────────────────
    _categorias: List[str] = []
    _catalogo:   Dict[str, Dict] = {}   # {codigo_producto: {nombre, aliases, stock, ...}}

    # ── Conexión ─────────────────────────────────────────────────────────────
    def validar_conexion(self) -> bool:
        """Comprueba que la BD es accesible. Devuelve True/False."""
        return probar_conexion()

    def _query(self, sql: str, params=None) -> List[Dict]:
        """Ejecuta una SELECT y devuelve lista de dicts. Uso interno."""
        with get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql, params)
                return [dict(row) for row in cur.fetchall()]

    def _execute(self, sql: str, params=None) -> None:
        """Ejecuta INSERT/UPDATE sin devolver filas. Uso interno."""
        with get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, params)
            conn.commit()

    # ── Validación de esquema ─────────────────────────────────────────────────
    def obtener_esquema_tablas(self) -> Dict[str, List[str]]:
        """
        Consulta information_schema para obtener columnas reales de cada tabla.
        Útil para detectar diferencias entre lo esperado y lo desplegado.
        """
        sql = """
            SELECT table_name, column_name
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name = ANY(%s)
            ORDER BY table_name, ordinal_position
        """
        rows = self._query(sql, (TABLAS_REQUERIDAS,))
        esquema: Dict[str, List[str]] = {}
        for row in rows:
            esquema.setdefault(row["table_name"], []).append(row["column_name"])
        return esquema

    def validar_contrato_minimo(self) -> bool:
        """
        Verifica que todas las tablas requeridas existen en la BD.
        Imprime un reporte y devuelve True si todo está OK.
        """
        esquema = self.obtener_esquema_tablas()
        ok = True
        print("── Validación de esquema ────────────────────────")
        for tabla in TABLAS_REQUERIDAS:
            if tabla in esquema:
                print(f"  ✓ {tabla}  ({len(esquema[tabla])} columnas)")
            else:
                print(f"  ✗ {tabla}  — TABLA NO ENCONTRADA")
                ok = False
        print("─────────────────────────────────────────────────")
        if ok:
            print("✓ Contrato mínimo cumplido.")
        else:
            print("✗ Hay tablas faltantes. Revisa el script SQL de creación.")
        return ok

    # ── Caché de catálogo ─────────────────────────────────────────────────────
    def refresh_cache(self) -> None:
        """
        Recarga categorías y catálogo de productos desde PostgreSQL.
        Llámalo al inicio y después de cualquier cambio en productos/inventario.
        """
        # Categorías
        rows = self._query(
            "SELECT nombre FROM categorias WHERE activo = TRUE ORDER BY nombre"
        )
        self._categorias = [r["nombre"] for r in rows]

        # Catálogo completo con stock
        rows = self._query("""
            SELECT
                p.codigo_producto,
                p.nombre,
                p.precio,
                p.aliases,
                p.activo,
                c.nombre AS categoria,
                i.stock_actual,
                i.stock_minimo
            FROM productos p
            JOIN categorias c ON c.categoria_id = p.categoria_id
            JOIN inventario i ON i.producto_id  = p.producto_id
            WHERE p.activo = TRUE
        """)

        self._catalogo = {}
        for r in rows:
            aliases_raw = r["aliases"] or ""
            aliases = [a.strip() for a in aliases_raw.split("|") if a.strip()]
            nombre = r["nombre"].strip()
            if nombre not in aliases:
                aliases.insert(0, nombre)
            self._catalogo[r["codigo_producto"]] = {
                "nombre":      nombre,
                "categoria":   r["categoria"],
                "precio":      float(r["precio"]),
                "stock":       int(r["stock_actual"]),
                "stock_minimo": int(r["stock_minimo"]),
                "aliases":     aliases,
            }

        print(f"✓ Caché recargada: {len(self._categorias)} categorías, "
              f"{len(self._catalogo)} productos")

    # ── Categorías ────────────────────────────────────────────────────────────
    def listar_categorias(self) -> List[str]:
        """Devuelve lista de categorías activas."""
        return list(self._categorias)

    def buscar_categoria(self, texto: str) -> Optional[str]:
        """
        Busca si el texto del usuario menciona alguna categoría existente.
        Devuelve el nombre de la categoría o None.
        """
        texto_n = _normalizar(texto)
        for cat in self._categorias:
            if _normalizar(cat) in texto_n:
                return cat
        return None

    # ── Productos ─────────────────────────────────────────────────────────────
    def buscar_producto(self, texto: str) -> Optional[str]:
        """
        Busca un producto en el catálogo usando aliases.
        Devuelve el codigo_producto o None.
        Prioridad: alias exacto > alias parcial.
        """
        texto_n = _normalizar(texto)
        # Evitar falsos positivos de categoría
        if self.buscar_categoria(texto):
            return None
        for codigo, datos in self._catalogo.items():
            for alias in datos["aliases"]:
                if _normalizar(alias) in texto_n:
                    return codigo
        return None

    def consultar_productos_por_categoria(self, categoria: str) -> List[Dict]:
        """
        Devuelve productos disponibles (stock > 0) de una categoría.
        Usa la función SQL fn_productos_categoria.
        """
        rows = self._query(
            "SELECT * FROM fn_productos_categoria(%s)",
            (categoria,)
        )
        return rows

    def consultar_stock_producto(self, codigo_producto: str) -> Optional[Dict]:
        """Devuelve el stock actual de un producto por su código."""
        rows = self._query(
            "SELECT * FROM v_stock_actual WHERE codigo_producto = %s",
            (codigo_producto,)
        )
        return rows[0] if rows else None

    # ── Clientes ─────────────────────────────────────────────────────────────
    def consultar_cliente(self, codigo_cliente: str) -> Optional[Dict]:
        """Devuelve un cliente por su código visible."""
        rows = self._query(
            "SELECT * FROM clientes WHERE codigo_cliente = %s",
            (codigo_cliente.upper(),)
        )
        return rows[0] if rows else None

    def _normalizar_nombre_cliente(self, nombre: str) -> str:
        """Normaliza nombre para comparar clientes sin mayúsculas ni espacios extra."""
        return re.sub(r"\s+", " ", nombre.strip().lower())

    def buscar_cliente_por_nombre(self, nombre: str) -> Optional[Dict]:
        """Busca un cliente por nombre normalizado para mitigar duplicados."""
        nombre_n = self._normalizar_nombre_cliente(nombre)
        rows = self._query("""
            SELECT *
            FROM clientes
            WHERE LOWER(REGEXP_REPLACE(TRIM(COALESCE(nombre, '')), '\\s+', ' ', 'g')) = %s
            ORDER BY codigo_cliente
            LIMIT 1
        """, (nombre_n,))
        return rows[0] if rows else None

    def obtener_o_crear_cliente_por_nombre(self, nombre: str) -> Dict:
        """Devuelve cliente existente por nombre o crea uno con el siguiente CLI-XXX."""
        nombre_limpio = re.sub(r"\s+", " ", nombre.strip())
        existente = self.buscar_cliente_por_nombre(nombre_limpio)
        if existente:
            existente["creado"] = False
            return existente

        rows = self._query("""
            SELECT COALESCE(MAX((REGEXP_MATCH(codigo_cliente, '^CLI-(\\d+)$'))[1]::INT), 0) + 1 AS siguiente
            FROM clientes
        """)
        codigo_cliente = f"CLI-{rows[0]['siguiente']:03d}"

        rows = self._query("""
            INSERT INTO clientes (codigo_cliente, nombre)
            VALUES (%s, %s)
            RETURNING *
        """, (codigo_cliente, nombre_limpio))
        cliente = rows[0]
        cliente["creado"] = True
        return cliente

    # ── Pedidos ───────────────────────────────────────────────────────────────
    def consultar_pedido(self, codigo_pedido: str) -> Optional[Dict]:
        """Devuelve la cabecera de un pedido por su código visible."""
        rows = self._query(
            "SELECT * FROM v_pedidos_resumen WHERE codigo_pedido = %s",
            (codigo_pedido.upper(),)
        )
        return rows[0] if rows else None

    def consultar_detalle_pedido(self, codigo_pedido: str) -> List[Dict]:
        """Devuelve las líneas del pedido con nombre de producto y subtotales."""
        return self._query(
            "SELECT * FROM v_detalle_pedido_completo WHERE codigo_pedido = %s",
            (codigo_pedido.upper(),)
        )

    def consultar_movimientos_pedido(self, codigo_pedido: str) -> List[Dict]:
        """Devuelve los movimientos de inventario asociados a un pedido."""
        return self._query("""
            SELECT
                m.fecha_hora,
                m.tipo_movimiento,
                p.nombre AS producto,
                m.cantidad,
                m.stock_anterior,
                m.stock_nuevo,
                m.motivo
            FROM movimientos_inventario m
            JOIN pedidos  pe ON pe.pedido_id  = m.pedido_id
            JOIN productos p ON p.producto_id = m.producto_id
            WHERE pe.codigo_pedido = %s
            ORDER BY m.fecha_hora
        """, (codigo_pedido.upper(),))

    # ── Operaciones de escritura ───────────────────────────────────────────────
    def crear_pedido(
        self,
        codigo_cliente: str,
        carrito: List[Dict],   # [{"codigo_producto": "PROD-001", "cantidad": 2}]
        canal: str = "chatbot",
        metodo_pago: Optional[str] = None,
        observaciones: Optional[str] = None,
    ) -> Dict:
        """
        Crea un pedido completo:
          1. Valida stock de cada línea.
          2. Genera codigo_pedido secuencial en PostgreSQL.
          3. Llama a fn_crear_pedido y fn_registrar_compra por cada línea.
        Devuelve {"ok": bool, "codigo_pedido": str, "errores": [...]}
        """
        errores = []
        # 1. Validar stock de todo el carrito antes de crear nada
        for item in carrito:
            rows = self._query(
                "SELECT * FROM fn_validar_stock(%s, %s)",
                (item["codigo_producto"], item["cantidad"])
            )
            if rows and not rows[0]["disponible"]:
                errores.append(rows[0]["mensaje"])

        if errores:
            return {"ok": False, "codigo_pedido": None, "errores": errores}

        # 2. Generar código de pedido (usando secuencia PostgreSQL)
        rows = self._query(
            "SELECT 'PED-' || LPAD(nextval('seq_pedidos')::TEXT, 3, '0') AS codigo"
        )
        codigo_pedido = rows[0]["codigo"]

        # 3. Crear cabecera
        rows = self._query(
            "SELECT * FROM fn_crear_pedido(%s, %s, %s, %s, %s)",
            (codigo_pedido, codigo_cliente, canal, metodo_pago, observaciones)
        )
        if not rows or not rows[0]["exito"]:
            return {"ok": False, "codigo_pedido": None,
                    "errores": [rows[0]["mensaje"] if rows else "Error desconocido"]}

        # 4. Registrar compra por cada línea
        for item in carrito:
            self._query(
                "SELECT * FROM fn_registrar_compra(%s, %s, %s)",
                (codigo_pedido, item["codigo_producto"], item["cantidad"])
            )

            # Añadir línea en detalle_pedidos
            self._execute("""
                INSERT INTO detalle_pedidos
                    (pedido_id, producto_id, cantidad_comprada, precio_unitario, estado_linea)
                SELECT pe.pedido_id, pr.producto_id, %s, pr.precio, 'pendiente'
                FROM pedidos pe, productos pr
                WHERE pe.codigo_pedido = %s AND pr.codigo_producto = %s
            """, (item["cantidad"], codigo_pedido, item["codigo_producto"]))

        # Refrescar caché de stock
        self.refresh_cache()
        return {"ok": True, "codigo_pedido": codigo_pedido, "errores": []}

    def registrar_devolucion(
        self,
        codigo_pedido: str,
        codigo_producto: str,
        cantidad: int,
        motivo: str = "devolucion cliente",
    ) -> Dict:
        """Registra una devolución y suma stock. Devuelve resultado de la función SQL."""
        rows = self._query(
            "SELECT * FROM fn_registrar_devolucion(%s, %s, %s, %s)",
            (codigo_pedido.upper(), codigo_producto, cantidad, motivo)
        )
        self.refresh_cache()
        return rows[0] if rows else {"exito": False, "mensaje": "Error desconocido"}

    # ── Logging ───────────────────────────────────────────────────────────────
    def guardar_log_chatbot(
        self,
        pregunta_cliente: str,
        intencion_detectada: str,
        confianza: float,
        origen_intencion: str,
        respuesta_bot: str,
        intencion_correcta: str = "",
        respuesta_correcta: str = "",
        aprobado_para_entrenamiento: bool = False,
        notas: str = "",
    ) -> None:
        """Inserta un registro en logs_chatbot en PostgreSQL."""
        self._execute("""
            INSERT INTO logs_chatbot (
                pregunta_cliente, intencion_detectada, intencion_correcta,
                confianza, origen_intencion, respuesta_bot, respuesta_correcta,
                aprobado_para_entrenamiento, notas
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            pregunta_cliente, intencion_detectada, intencion_correcta,
            round(confianza, 4), origen_intencion, respuesta_bot, respuesta_correcta,
            aprobado_para_entrenamiento, notas,
        ))


### Inicializar RetailService

In [5]:
# ── Inicialización ───────────────────────────────────────────────────────────
# Si la conexión falla, el chatbot seguirá funcionando en modo degradado
# usando reglas de intención y respondiendo sin datos de inventario.

service = RetailService()
if service.validar_conexion():
    service.validar_contrato_minimo()
    service.refresh_cache()
else:
    print("⚠ Chatbot en modo degradado — sin conexión a PostgreSQL")
    print("  Revisa las credenciales en tu archivo .env")


✓ Conexión OK → PostgreSQL 18.0 on x86_64-windows, compiled by msvc-19.44.35
── Validación de esquema ────────────────────────
  ✓ categorias  (5 columnas)
  ✓ productos  (10 columnas)
  ✓ inventario  (5 columnas)
  ✓ clientes  (6 columnas)
  ✓ pedidos  (10 columnas)
  ✓ detalle_pedidos  (8 columnas)
  ✓ movimientos_inventario  (9 columnas)
  ✓ logs_chatbot  (11 columnas)
─────────────────────────────────────────────────
✓ Contrato mínimo cumplido.
✓ Caché recargada: 9 categorías, 12 productos


## 4. Utilidades NLP

In [6]:
# ============================================================
# 4. UTILIDADES NLP
# Funciones de texto que usa el motor de intenciones.
# No contienen SQL; leen del caché de RetailService.
# ============================================================

def _normalizar(texto: str) -> str:
    """Pasa a minúsculas, elimina acentos y espacios extra."""
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def extraer_id_pedido(mensaje: str) -> Optional[str]:
    """Extrae un código tipo PED-123 del mensaje del usuario."""
    match = re.search(r"\bped[-\s]?\d+\b", mensaje, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group().replace(" ", "-").upper()


def extraer_codigo_cliente(mensaje: str) -> Optional[str]:
    """Extrae un código tipo CLI-001 del mensaje del usuario."""
    match = re.search(r"\bcli[-\s]?\d+\b", mensaje, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group().replace(" ", "-").upper()


def extraer_cantidad_solicitada(mensaje: str) -> int:
    """Extrae la cantidad numérica solicitada o devuelve CANTIDAD_POR_DEFECTO."""
    msg_n = _normalizar(mensaje)
    patrones = [
        r"\b(?:quiero|necesito|comprar|llevar|pedir|solicito)\s+(\d+)\b",
        r"\b(\d+)\s+(?:unidades|uds|piezas|kilos|kg|paquetes)?\b",
    ]
    for patron in patrones:
        m = re.search(patron, msg_n)
        if m:
            cantidad = int(m.group(1))
            if cantidad > 0:
                return cantidad
    return CANTIDAD_POR_DEFECTO


def tiene_cantidad_explicita(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return bool(re.search(r"\b\d+\b", msg_n))


def contiene_alguna(texto_n: str, expresiones: List[str]) -> bool:
    return any(e in texto_n for e in expresiones)


def es_respuesta_afirmativa(mensaje: str) -> bool:
    return _normalizar(mensaje) in {"si", "sí", "claro", "vale", "ok", "dale", "por favor", "exacto", "confirmo", "confirmar"}


def es_respuesta_negativa(mensaje: str) -> bool:
    return _normalizar(mensaje) in {"no", "nop", "cancelar", "cancela", "mejor no", "anular"}


def es_solicitud_compra(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["comprar", "llevar", "pedir", "solicito", "agregar", "añadir", "anadir", "quiero", "necesito"])


def parece_nombre_completo(mensaje: str) -> bool:
    partes = re.sub(r"\s+", " ", mensaje.strip()).split(" ")
    return len([p for p in partes if len(p) >= 2]) >= 2


def necesita_soporte_por_palabras_clave(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, [
        "humano", "agente", "persona", "reclamo", "queja", "urgente",
        "denuncia", "no me ayudan"
    ])


def quiere_ver_categorias(mensaje: str) -> bool:
    msg_n = _normalizar(mensaje)
    return contiene_alguna(msg_n, ["categoria", "categorias", "otras categorias", "ver categorias"])


## 5. Reglas de intención

In [7]:
# ============================================================
# 5. REGLAS DE INTENCION + CLASIFICADOR ZERO-SHOT
# La detección por reglas tiene prioridad sobre el modelo.
# ============================================================

def detectar_intencion_por_reglas(mensaje: str) -> Optional[str]:
    msg_n = _normalizar(mensaje)

    if necesita_soporte_por_palabras_clave(mensaje):
        return INTENCION_SOPORTE

    if contiene_alguna(msg_n, ["caduc", "vencid", "mal estado", "danad", "roto", "podrid"]):
        return "Producto dañado caducado o en mal estado"

    if contiene_alguna(msg_n, [
        "no ha llegado", "no llego", "no recibi", "incompleto",
        "faltan", "falta", "equivocado", "entrega tarde", "retraso"
    ]):
        return "Incidencia con pedido incompleto o no recibido"

    if contiene_alguna(msg_n, ["devolver", "devolucion", "cambiar", "cambio", "reembolso"]):
        return "Devoluciones y cambios"

    if contiene_alguna(msg_n, ["cupon", "promocion", "descuento", "oferta", "puntos"]):
        return "Promociones y cupones"

    if contiene_alguna(msg_n, ["tarjeta", "pago", "pagar", "checkout", "paypal", "bizum"]):
        return "Métodos de pago y compra"

    if quiere_ver_categorias(mensaje) and not service.buscar_categoria(mensaje):
        return "Disponibilidad de productos"

    if service.buscar_categoria(mensaje) and contiene_alguna(msg_n, [
        "disponible", "stock", "inventario", "tienen", "hay",
        "categoria", "categorias", "productos"
    ]):
        return "Disponibilidad de productos"

    if service.buscar_producto(mensaje) and (
        contiene_alguna(msg_n, [
            "disponible", "stock", "inventario", "tienen", "hay",
            "comprar", "disponibilidad", "quiero", "necesito", "llevar", "pedir"
        ])
        or extraer_cantidad_solicitada(mensaje) > CANTIDAD_POR_DEFECTO
    ):
        return "Disponibilidad de productos"

    if extraer_id_pedido(mensaje) or contiene_alguna(msg_n, [
        "estado del pedido", "donde esta mi pedido", "seguimiento"
    ]):
        return "Estado del pedido"

    return None


# ── Carga del modelo zero-shot ────────────────────────────────────────────────
import os
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")

def cargar_clasificador_zero_shot():
    print("Cargando modelo NLP (mDeBERTa)... esto puede tardar unos segundos.")
    try:
        from transformers import pipeline as hf_pipeline
        clf = hf_pipeline(
            "zero-shot-classification",
            model=MODEL_NAME,
            tokenizer=MODEL_NAME,
            use_fast=False,
            local_files_only=MODEL_LOCAL_ONLY,
        )
        print("✓ Modelo cargado.")
        return clf
    except Exception as exc:
        print(f"⚠ No se pudo cargar el modelo: {exc}")
        print("  El chatbot usará solo reglas de intención.")
        return None

clasificador = cargar_clasificador_zero_shot()


def procesar_mensaje(mensaje: str):
    """
    Determina la intención del usuario.
    Primero aplica reglas; si no hay match, usa el modelo zero-shot.
    Devuelve: (intencion, score, ranking, origen)
    """
    intencion_regla = detectar_intencion_por_reglas(mensaje)
    if intencion_regla:
        return intencion_regla, 1.0, [(intencion_regla, 1.0)], "regla"

    if clasificador is None:
        return INTENCION_SOPORTE, 0.0, [(INTENCION_SOPORTE, 0.0)], "modelo_no_disponible"

    resultado = clasificador(mensaje, ETIQUETAS_NEGOCIO)
    ranking = list(zip(resultado["labels"], resultado["scores"]))
    intencion = resultado["labels"][0]
    score = resultado["scores"][0]

    if score < CONFIDENCE_THRESHOLD:
        return INTENCION_SOPORTE, score, ranking, "modelo_zero_shot_baja_confianza"

    return intencion, score, ranking, "modelo_zero_shot"


Cargando modelo NLP (mDeBERTa)... esto puede tardar unos segundos.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo cargado.


## 6. Generación de respuestas

In [8]:
# ============================================================
# 6. GENERACION DE RESPUESTAS
# Todas las funciones usan service.* — cero SQL directo aquí.
# ============================================================

@dataclass
class EstadoConversacion:
    esperando_id_pedido:              bool = False
    esperando_confirmacion_categorias: bool = False
    esperando_nombre_cliente:         bool = False
    esperando_confirmacion_pedido:    bool = False
    ultima_intencion:                 Optional[str] = None
    pedido_pendiente:                 Optional[Dict[str, Any]] = None


# ── Formateo de detalle y movimientos ────────────────────────────────────────

def _formatear_detalle(lineas: List[Dict]) -> str:
    if not lineas:
        return ""
    partes = []
    for ln in lineas:
        texto = (
            f'{ln["producto"]}: comprado {ln["cantidad_comprada"]}, '
            f'entregado {ln["cantidad_entregada"]}, estado {ln["estado_linea"]}'
        )
        if ln.get("motivo_incidencia"):
            texto += f' ({ln["motivo_incidencia"]})'
        partes.append(texto)
    return " Detalle: " + "; ".join(partes) + "."


def _formatear_movimientos(movimientos: List[Dict]) -> str:
    if not movimientos:
        return ""
    compras = sum(m["cantidad"] for m in movimientos if m["tipo_movimiento"] == "compra")
    devols  = sum(m["cantidad"] for m in movimientos if m["tipo_movimiento"] == "devolucion")
    partes  = []
    if compras:
        partes.append(f"compras registradas: {compras} unidades")
    if devols:
        partes.append(f"devoluciones registradas: {devols} unidades")
    return (" Movimientos de inventario: " + ", ".join(partes) + ".") if partes else ""


# ── Respuestas por intención ──────────────────────────────────────────────────

def responder_estado_pedido(mensaje: str, estado: EstadoConversacion) -> str:
    codigo = extraer_id_pedido(mensaje)

    if not codigo:
        estado.esperando_id_pedido = True
        return "Claro, puedo revisar tu pedido. Indícame la referencia, por ejemplo PED-123."

    estado.esperando_id_pedido = False
    pedido = service.consultar_pedido(codigo)

    if not pedido:
        return f"No encuentro el pedido {codigo}. Revisa la referencia o te derivo con soporte."

    detalle   = _formatear_detalle(service.consultar_detalle_pedido(codigo))
    movs      = _formatear_movimientos(service.consultar_movimientos_pedido(codigo))

    return (
        f"Tu pedido {codigo} está en estado: {pedido['estado_pedido']}. "
        f"Entrega estimada: {pedido['fecha_entrega_estimada']}."
        f"{detalle}{movs}"
    )


def responder_incidencia_pedido(mensaje: str) -> str:
    codigo = extraer_id_pedido(mensaje)
    if codigo and service.consultar_pedido(codigo):
        detalle = _formatear_detalle(service.consultar_detalle_pedido(codigo))
        return (
            f"Lamento la incidencia con el pedido {codigo}.{detalle} "
            "Si hay productos faltantes o en mal estado, puedo ayudarte a preparar la reclamación."
        )
    if codigo:
        return (
            f"Lamento la incidencia con el pedido {codigo}. "
            "Dime si faltan productos, llegó tarde o hubo un error en la entrega."
        )
    return (
        "Lamento lo ocurrido con tu pedido. Indícame el número de pedido "
        "y si no llegó, llegó incompleto o contiene productos equivocados."
    )


def _describir_pedido_pendiente(estado: EstadoConversacion) -> str:
    pedido = estado.pedido_pendiente or {}
    cantidad = pedido.get("cantidad")
    unidad = "unidad" if cantidad == 1 else "unidades"
    return f"{cantidad} {unidad} de {pedido.get('nombre_producto')}"


def _limpiar_pedido_pendiente(estado: EstadoConversacion) -> None:
    estado.esperando_nombre_cliente = False
    estado.esperando_confirmacion_pedido = False
    estado.pedido_pendiente = None


def _formatear_pedido_creado(codigo_pedido: str) -> str:
    pedido = service.consultar_pedido(codigo_pedido)
    detalle = _formatear_detalle(service.consultar_detalle_pedido(codigo_pedido))
    if not pedido:
        return f"Pedido creado correctamente. Tu código de seguimiento es {codigo_pedido}."
    entrega = pedido.get("fecha_entrega_estimada") or "pendiente de asignar"
    return (
        f"Pedido creado correctamente. Tu código de seguimiento es {codigo_pedido}. "
        f"Estado: {pedido['estado_pedido']}. Entrega estimada: {entrega}."
        f"{detalle}"
    )


def iniciar_flujo_compra(codigo_producto: str, cantidad: int, estado: EstadoConversacion) -> str:
    datos = service._catalogo.get(codigo_producto)
    if not datos:
        return "No encontré información de ese producto en el sistema."

    estado.pedido_pendiente = {
        "codigo_producto": codigo_producto,
        "nombre_producto": datos["nombre"],
        "cantidad": cantidad,
    }
    estado.esperando_nombre_cliente = True
    estado.esperando_confirmacion_pedido = False
    unidad = "unidad" if cantidad == 1 else "unidades"
    return (
        f"Puedo preparar el pedido de {cantidad} {unidad} de {datos['nombre']}. "
        "Indícame tu nombre completo, por ejemplo: María Gómez."
    )


def responder_nombre_cliente_para_pedido(mensaje: str, estado: EstadoConversacion) -> str:
    if not parece_nombre_completo(mensaje):
        return "Para crear el pedido necesito tu nombre y apellido, por ejemplo: María Gómez."

    nombre_limpio = re.sub(r"\s+", " ", mensaje.strip())
    cliente = service.buscar_cliente_por_nombre(nombre_limpio)
    estado.pedido_pendiente["nombre_cliente"] = cliente.get("nombre") if cliente else nombre_limpio
    estado.pedido_pendiente["codigo_cliente"] = cliente["codigo_cliente"] if cliente else None
    estado.pedido_pendiente["cliente_existente"] = bool(cliente)
    estado.esperando_nombre_cliente = False
    estado.esperando_confirmacion_pedido = True
    accion_cliente = (
        f"encontré el cliente {cliente['codigo_cliente']}" if cliente
        else "crearé un cliente nuevo al confirmar"
    )
    return (
        f"Perfecto, {accion_cliente} para {estado.pedido_pendiente['nombre_cliente']}. "
        f"Voy a crear un pedido con {_describir_pedido_pendiente(estado)}. "
        "Confírmame con 'sí' para generarlo o 'no' para cancelarlo."
    )


def confirmar_pedido_pendiente(mensaje: str, estado: EstadoConversacion) -> str:
    if es_respuesta_negativa(mensaje):
        _limpiar_pedido_pendiente(estado)
        return "Listo, cancelé la creación del pedido."

    if not es_respuesta_afirmativa(mensaje):
        return "Confírmame con 'sí' para crear el pedido o 'no' para cancelarlo."

    pedido = estado.pedido_pendiente or {}
    if not pedido.get("codigo_cliente"):
        cliente = service.obtener_o_crear_cliente_por_nombre(pedido["nombre_cliente"])
        pedido["codigo_cliente"] = cliente["codigo_cliente"]

    resultado = service.crear_pedido(
        codigo_cliente=pedido["codigo_cliente"],
        carrito=[{"codigo_producto": pedido["codigo_producto"], "cantidad": pedido["cantidad"]}],
        canal="chatbot",
        observaciones="Pedido creado desde conversación del chatbot",
    )
    _limpiar_pedido_pendiente(estado)

    if not resultado["ok"]:
        return "No pude crear el pedido: " + "; ".join(resultado["errores"])

    return _formatear_pedido_creado(resultado["codigo_pedido"])


def responder_disponibilidad(mensaje: str, estado: EstadoConversacion) -> str:
    # 1. El usuario quiere ver categorías directamente
    if quiere_ver_categorias(mensaje) and not service.buscar_categoria(mensaje):
        estado.esperando_confirmacion_categorias = False
        cats = service.listar_categorias()
        if not cats:
            return "No tengo categorías disponibles en este momento."
        return "Estas son las categorías disponibles: " + ", ".join(cats) + "."

    # 2. El usuario menciona una categoría
    categoria = service.buscar_categoria(mensaje)
    if categoria:
        estado.esperando_confirmacion_categorias = True
        prods = service.consultar_productos_por_categoria(categoria)
        disponibles = [p for p in prods if p["disponible"]]
        if not disponibles:
            return f"Ahora mismo no tengo productos disponibles en la categoría {categoria}."
        lista = ", ".join(f'{p["nombre"]} ({p["stock_actual"]} uds)' for p in disponibles)
        return (
            f"En la categoría {categoria} tengo disponibles: {lista}. "
            "¿Deseas conocer otras categorías o productos?"
        )

    # 3. El usuario menciona un producto concreto
    codigo = service.buscar_producto(mensaje)
    if not codigo:
        return (
            "Puedo consultar disponibilidad, pero necesito el producto exacto o la categoría. "
            "Por ejemplo: manzanas, pan sin gluten o categoría lácteos."
        )

    datos = service._catalogo.get(codigo)
    if not datos:
        return "No encontré información de ese producto en el sistema."

    stock    = datos["stock"]
    nombre   = datos["nombre"]
    cantidad = extraer_cantidad_solicitada(mensaje)

    if stock <= 0:
        return (
            f"Ahora mismo {nombre} aparece agotado en el inventario online. "
            "Puedes revisar más tarde o consultar alternativas."
        )
    if cantidad > stock:
        return (
            f"Ahora mismo solo hay {stock} unidades de {nombre} disponibles. "
            f"No puedo confirmar una compra de {cantidad} unidades con el inventario actual. "
            "Puedes reducir la cantidad o consultar alternativas."
        )
    if tiene_cantidad_explicita(mensaje) and es_solicitud_compra(mensaje):
        return iniciar_flujo_compra(codigo, cantidad, estado)

    if cantidad > CANTIDAD_POR_DEFECTO:
        return (
            f"Sí, hay stock suficiente para {cantidad} unidades de {nombre}. "
            f"Actualmente figuran {stock} unidades en el inventario online."
        )
    return (
        f"Sí, actualmente figuran {stock} unidades de {nombre} en el inventario online. "
        "La disponibilidad puede variar al finalizar la compra."
    )


def responder_producto_mal_estado(mensaje: str) -> str:
    codigo = service.buscar_producto(mensaje)
    if codigo:
        nombre = service._catalogo[codigo]["nombre"]
        return (
            f"Siento que hayas recibido {nombre} en mal estado. "
            "Conserva el ticket o número de pedido y, si puedes, una foto. "
            "Te derivo con atención al cliente."
        )
    return (
        "Siento que hayas recibido un producto en mal estado. "
        "Indícame el producto, el número de pedido y si tienes foto o ticket."
    )


def obtener_respuesta(intencion: str, mensaje: str, estado: EstadoConversacion) -> str:
    """Enrutador principal de respuestas."""

    if estado.pedido_pendiente and es_respuesta_negativa(mensaje):
        _limpiar_pedido_pendiente(estado)
        return "Listo, cancelé la creación del pedido."

    # Contexto previo: esperando nombre completo para crear/buscar cliente
    if estado.esperando_nombre_cliente:
        return responder_nombre_cliente_para_pedido(mensaje, estado)

    # Contexto previo: esperando confirmación final del pedido
    if estado.esperando_confirmacion_pedido:
        return confirmar_pedido_pendiente(mensaje, estado)

    # Contexto previo: esperando confirmación de listar categorías
    if estado.esperando_confirmacion_categorias and es_respuesta_afirmativa(mensaje):
        estado.esperando_confirmacion_categorias = False
        cats = service.listar_categorias()
        return "Estas son las categorías disponibles: " + ", ".join(cats) + "."

    # Contexto previo: esperando código de pedido
    if estado.esperando_id_pedido and extraer_id_pedido(mensaje):
        return responder_estado_pedido(mensaje, estado)

    estado.ultima_intencion = intencion

    if intencion == "Estado del pedido":
        return responder_estado_pedido(mensaje, estado)
    if intencion == "Incidencia con pedido incompleto o no recibido":
        return responder_incidencia_pedido(mensaje)
    if intencion == "Disponibilidad de productos":
        return responder_disponibilidad(mensaje, estado)
    if intencion == "Producto dañado caducado o en mal estado":
        return responder_producto_mal_estado(mensaje)
    if intencion == "Devoluciones y cambios":
        return POLITICAS["devoluciones"] + " Si quieres, puedo ayudarte a preparar la solicitud."
    if intencion == "Promociones y cupones":
        return POLITICAS["promociones"] + " Si un cupón no funciona, revisa fecha de validez e importe mínimo."
    if intencion == "Métodos de pago y compra":
        return POLITICAS["pagos"]
    if intencion == INTENCION_SOPORTE:
        return "Quiero evitar darte una respuesta incorrecta. Te puedo derivar con atención al cliente; antes, cuéntame brevemente qué ocurrió."

    return "No he entendido bien tu consulta. Puedo ayudarte con pedidos, devoluciones, productos, promociones o soporte."


## 7. Logging del chatbot
Cada interacción se guarda en `logs_chatbot` en PostgreSQL para reentrenamiento.

In [9]:
# ============================================================
# 7. LOGGING A POSTGRESQL
# ============================================================

def registrar_interaccion(
    mensaje: str,
    intencion: str,
    score: float,
    origen: str,
    respuesta: str,
) -> None:
    """
    Guarda la interacción en logs_chatbot.
    Si falla (sin conexión), avisa pero no interrumpe el chat.
    """
    try:
        service.guardar_log_chatbot(
            pregunta_cliente=mensaje,
            intencion_detectada=intencion,
            confianza=score,
            origen_intencion=origen,
            respuesta_bot=respuesta,
        )
    except Exception as exc:
        print(f"  ⚠ No se pudo guardar el log: {exc}")


## 8. Tests conversacionales
Ejecuta esta celda para validar el pipeline completo contra los datos reales de PostgreSQL.

In [10]:
# ============================================================
# 8. TEST SUITE AUTOMATICO
# ============================================================

TEST_QUERIES = [
    # Disponibilidad de categoría
    "qué productos de la categoría lácteos tienes disponible",
    # Confirmación afirmativa → listar categorías
    "sí",
    # Compra con cantidad
    "quiero comprar 100 manzanas",
    # Compra válida
    "necesito 6 panes sin gluten",
    # Estado de pedido
    "dónde está mi pedido PED-123",
    # Incidencia
    "mi pedido PED-901 llegó incompleto",
    # Producto mal estado
    "recibí un yogur caducado",
    # Devolución
    "quiero devolver un producto",
    # Pago
    "aceptan tarjeta de crédito",
    # Soporte
    "quiero hablar con una persona",
    # Pedido sin stock
    "quiero comprar 500 manzanas",
]

def ejecutar_tests():
    estado = EstadoConversacion()
    print("\n" + "=" * 65)
    print(f"  TESTS ChatBot EcoMarket v{VERSION} — fuente: PostgreSQL")
    print("=" * 65)

    for query in TEST_QUERIES:
        intencion, score, ranking, origen = procesar_mensaje(query)
        respuesta = obtener_respuesta(intencion, query, estado)

        print(f"\nUsuario : {query}")
        print(f"Intención: {intencion}  (score={score:.2f}, origen={origen})")
        print(f"Bot     : {respuesta}")
        print("-" * 65)

ejecutar_tests()



  TESTS ChatBot EcoMarket v1.1.2 — fuente: PostgreSQL

Usuario : qué productos de la categoría lácteos tienes disponible
Intención: Disponibilidad de productos  (score=1.00, origen=regla)
Bot     : En la categoría lacteos tengo disponibles: leche entera (15 uds), mantequilla (12 uds), yogur (4 uds). ¿Deseas conocer otras categorías o productos?
-----------------------------------------------------------------

Usuario : sí
Intención: Hablar con soporte humano  (score=0.43, origen=modelo_zero_shot_baja_confianza)
Bot     : Estas son las categorías disponibles: bebidas, despensa, fruta, lacteos, limpieza, origen animal, panaderia, snacks, verdura.
-----------------------------------------------------------------

Usuario : quiero comprar 100 manzanas
Intención: Disponibilidad de productos  (score=1.00, origen=regla)
Bot     : Ahora mismo solo hay 49 unidades de manzanas disponibles. No puedo confirmar una compra de 100 unidades con el inventario actual. Puedes reducir la cantidad o cons

## 9. Chat interactivo

In [11]:
# ============================================================
# 9. CHAT INTERACTIVO
# Ejecuta esta celda para iniciar la sesión conversacional.
# Escribe 'salir' para terminar.
# ============================================================

def ejecutar_chat():
    estado = EstadoConversacion()

    print("\n" + "=" * 65)
    print(f"  CHATBOT ECOMARKET v{VERSION} — PostgreSQL")
    print("=" * 65)
    print("  Escribe 'salir' para terminar.\n")

    while True:
        texto = input("Usuario: ").strip()

        if not texto:
            continue

        if _normalizar(texto) in COMANDOS_SALIDA:
            print("Bot: Hasta pronto. Cerrando sesión.")
            break

        intencion, score, _, origen = procesar_mensaje(texto)
        respuesta = obtener_respuesta(intencion, texto, estado)
        registrar_interaccion(texto, intencion, score, origen, respuesta)

        print(f"Bot: {respuesta}")
        print(f"     [intención={intencion} | score={score:.2f} | origen={origen}]")
        print("-" * 65)

ejecutar_chat()



  CHATBOT ECOMARKET v1.1.2 — PostgreSQL
  Escribe 'salir' para terminar.

Bot: Puedo preparar el pedido de 2 unidades de manzanas. Indícame tu nombre completo, por ejemplo: María Gómez.
     [intención=Disponibilidad de productos | score=1.00 | origen=regla]
-----------------------------------------------------------------
Bot: Perfecto, crearé un cliente nuevo al confirmar para Esteban Orozco. Voy a crear un pedido con 2 unidades de manzanas. Confírmame con 'sí' para generarlo o 'no' para cancelarlo.
     [intención=Hablar con soporte humano | score=0.27 | origen=modelo_zero_shot_baja_confianza]
-----------------------------------------------------------------
Bot: No pude crear el pedido: El c?digo de pedido PED-100 ya existe
     [intención=Hablar con soporte humano | score=0.42 | origen=modelo_zero_shot_baja_confianza]
-----------------------------------------------------------------
Bot: Hasta pronto. Cerrando sesión.
